## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: TRANSFORMACIÓN (CAPA SILVER)
**OBJETIVO:** Limpiar, cruzar tablas (Tickets + Agentes) y aplicar reglas de negocio para enriquecer los datos.

In [0]:
from pyspark.sql.functions import col, current_timestamp, when
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
    
# Limpiamos widgets previos para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
# 1. PARÁMETROS DINÁMICOS (WIDGETS)

dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

print("✅ Parámetros cargados.")

In [0]:
# 2. LECTURA DE TABLAS DESDE BRONZE

df_tickets = spark.table(f"{catalogo}.{esquema_source}.tickets_raw")
df_agentes = spark.table(f"{catalogo}.{esquema_source}.agentes_soporte_raw")

In [0]:
display(df_tickets)

In [0]:
display(df_agentes)

In [0]:
# 3. CREACIÓN DE FUNCIÓN PERSONALIZADA (UDF)
# Regla de negocio: Categorizar la criticidad basada en palabras clave del cuerpo del mensaje

def evaluar_criticidad(mensaje):
    # Si el mensaje viene vacío o nulo, le asignamos Baja por defecto
    if not mensaje:
        return "Baja"
        
    mensaje_lower = mensaje.lower()
    
    # Palabras clave para criticidad Alta
    palabras_alta = ["urgente", "emergencia", "grave", "caído", "inmediato", "inmediata", "roto"]
    if any(palabra in mensaje_lower for palabra in palabras_alta):
        return "Alta"
        
    # Palabras clave para criticidad Media
    palabras_media = ["ayuda", "duda", "problema", "revisar", "falla", "error"]
    if any(palabra in mensaje_lower for palabra in palabras_media):
        return "Media"
        
    # Si no tiene palabras clave críticas, es Baja
    return "Baja"

criticidad_udf = F.udf(evaluar_criticidad, StringType())

In [0]:
from pyspark.sql.functions import col, trim, when

# 4. LIMPIEZA Y TRANSFORMACIÓN
# Nuevo radar basado 100% en los mensajes reales del JSON
palabras_criticas = "(?i)pésimo|robo|cancelar|grosero|doble|cierra sola"

# Aplicamos la limpieza y calculamos la criticidad
df_tickets_clean = df_tickets.dropna(how="any", subset=["ticket_id"]) \
    .withColumn("agente_id", trim(col("agente_id")).cast("int")) \
    .withColumn(
        "criticidad", 
        when(col("cuerpo_mensaje").rlike(palabras_criticas), 1).otherwise(0)
    )

In [0]:
# ==========================================
# 🚨 RADAR DE DIAGNÓSTICO PARA PRODUCCIÓN 🚨
# ==========================================
print("1. MUESTRA DE TICKETS (Capa Bronze):")
# Mostramos si realmente viene el agente_id desde la extracción
df_tickets_clean.select("ticket_id", "agente_id").show(5)

print("2. TOTAL DE TICKETS CON AGENTE ASIGNADO ANTES DEL JOIN:")
# Contamos cuántos tickets tienen un agente_id que NO sea nulo
con_agente = df_tickets_clean.filter(col("agente_id").isNotNull()).count()
print(f"Tickets listos para cruzar: {con_agente}")

print("3. MUESTRA DE AGENTES (Capa Bronze):")
# Mostramos cómo se llaman las columnas y qué id tienen los agentes
df_agentes.show(5)
# ==========================================

In [0]:
display(df_tickets_clean)

In [0]:
# 5. CRUCE DE TABLAS (JOIN)
from pyspark.sql.functions import col, trim

# 1. Estandarizar el nombre de la columna en agentes
if "id_agente" in df_agentes.columns:
    df_agentes = df_agentes.withColumnRenamed("id_agente", "agente_id")

# 2. LIMPIEZA EXTREMA: Quitamos espacios ocultos (trim) y forzamos a entero en AMBAS tablas
df_tickets_clean = df_tickets_clean.withColumn("agente_id", trim(col("agente_id")).cast("int"))
df_agentes = df_agentes.withColumn("agente_id", trim(col("agente_id")).cast("int"))

# 3. JOIN SEGURO: Al usar el string "agente_id", PySpark unifica las columnas 
# y evitamos que los alias (t.agente_id vs a.agente_id) causen columnas nulas
df_joined = df_tickets_clean.join(df_agentes, on="agente_id", how="left")

# ==========================================
# 🚨 RADAR FINAL POST-JOIN 🚨
# ==========================================
print("4. RESULTADO DESPUÉS DEL JOIN:")
df_joined.select("ticket_id", "agente_id", "nombre_completo", "region").show(5)

In [0]:
display(df_joined)

In [0]:
# 6. SELECCIÓN FINAL Y GUARDADO EN SILVER

tabla_destino = f"{catalogo}.{esquema_sink}.tickets_enriquecidos"

df_silver = df_joined.select(
    col("ticket_id"),
    col("cliente_id"),
    col("cuerpo_mensaje").alias("descripcion"),
    col("criticidad"),
    col("nombre_completo").alias("agente_asignado"),
    col("region").alias("region_agente"),
    col("nivel_soporte"),
    current_timestamp().alias("fecha_transformacion")
)

df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_destino)

print(f"✅ Transformación completada. Tabla guardada en {tabla_destino}")

In [0]:
display(tabla_destino)

In [0]:
display(df_silver)